# Debug: plot_umap package vs original pipeline

Runs both the packaged `tm.plot_umap()` and a verbatim inline copy of the original `load_sample_umap` / `compute_kde_grid` logic
on the **exact same sample, same reference map, same embeddings**, so any remaining difference is isolated to real logic
differences rather than different input data.

Both sides share:
- the same repertoire dataframe (`df`)
- the same reference map (loaded once via `tm.load_reference(MODEL_PATH)`)
- the same embedding dict (computed once via `tm.embed_repertoire(df)`, passed into both paths — this removes embedding
  computation as a variable, since we already verified it matches separately)

So if `n_clones` and coordinates still differ after this, the bug is in the downsampling/scoring logic itself, not the data feeding it.

In [ ]:
import os

# --- EDIT ME if needed: only one of these two should be set (or neither,
# if you're relying on an already-logged-in public/private HF Hub repo).
# os.environ["TCRMETA_HF_REPO"] = "yourname/tcrmeta-weights"
# os.environ["TCRMETA_WEIGHTS_DIR"] = "/path/to/local/weights/folder"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

import tcrmeta as tm

print("tcrmeta version:", tm.__version__)

## Parameters — edit these to match your original run

In [ ]:
SAMPLE_CSV = "/mnt/isilon/boli_lab/globus/project/mingyao/tcr_database/HIV/csv/015V08002889_CFAR.csv"

# Optional: precomputed embedding pkl for this same sample, if you have one
# from your original pipeline's EMB_DIR. If set and the file exists, we'll
# cross-check it against the embeddings the package computes fresh.
EMB_DIR = "/mnt/isilon/boli_lab/globus/project/mingyao/tcr_database/HIV/emb"

MODEL_PATH = (
    "/mnt/isilon/boli_lab/globus/project/mingyao/DeepCluster/distribution_shift/"
    "Tonon/reference_map/density_map_output/density_map_output_pca30_10000_thres/"
    "vgene_density_models.pkl"
)

V_GENE = "TRBV27"
TARGET_DEPTH = 10000
SEED = 42
GRID_N = 300

sample_id = os.path.splitext(os.path.basename(SAMPLE_CSV))[0]
print("sample_id:", sample_id)

## Step 1 — load the raw CSV and rename to tcrmeta's required columns

In [ ]:
df_raw = pd.read_csv(SAMPLE_CSV)

col_map = {
    "aminoAcid": "cdr3aa",
    "vGeneName": "v_gene",
    "count (templates/reads)": "count",
    "templates": "count",
    "productive_frequency": "freq",
}
df = df_raw.rename(columns={k: v for k, v in col_map.items() if k in df_raw.columns})
if "count" not in df.columns and "freq" in df.columns:
    df["count"] = (df["freq"].astype(float) * 1_000_000).round().astype(int)

df = df[["cdr3aa", "v_gene", "count"]].copy()
print(f"{len(df)} raw rows, {df['v_gene'].nunique()} unique V genes")
print(f"{V_GENE} raw row count: {(df['v_gene'] == V_GENE).sum()}")
df.head()

## Step 2 — load the reference map once, embed once (shared by both paths below)

In [ ]:
reference = tm.load_reference(MODEL_PATH)
print(f"Loaded reference: {len(reference)} V gene models, {V_GENE} present: {V_GENE in reference}")

embeddings = tm.embed_repertoire(df, device="auto")
print(f"Embedded {len(embeddings)} unique clones")

# Optional cross-check against a precomputed embedding pkl, if you have one
emb_pkl_path = os.path.join(EMB_DIR, f"{sample_id}.pkl")
if os.path.exists(emb_pkl_path):
    import pickle
    with open(emb_pkl_path, "rb") as f:
        emb_dict_orig = pickle.load(f)
    shared_keys = list(set(embeddings) & set(emb_dict_orig))
    if shared_keys:
        k = shared_keys[0]
        diff = np.abs(embeddings[k] - emb_dict_orig[k]).max()
        print(f"Cross-check vs precomputed pkl: {len(shared_keys)} shared clones, "
              f"max abs diff on one example = {diff:.2e}")
else:
    print(f"No precomputed embedding pkl found at {emb_pkl_path} — skipping cross-check (fine, not required).")

## Step 3 — package path: `tm.plot_umap`

In [ ]:
fig_pkg, coords_pkg = tm.plot_umap(
    df,
    reference=reference,
    v_gene=V_GENE,
    embeddings=embeddings,   # reuse the exact same embeddings computed above
    downsample=TARGET_DEPTH,
    seed=SEED,
    grid_n=GRID_N,
)
print(f"[package] {V_GENE}: {len(coords_pkg)} clones")
coords_pkg.head()

## Step 4 — original-logic path (verbatim inline copy, using the same `df` + `embeddings`)

This is your `load_sample_umap` function's body, adapted only to take `df`/`embeddings` directly instead of
re-reading from `CSV_DIR`/`EMB_DIR` — everything else (rename fallback, downsample, per-V-gene filter, scaler/UMAP transform)
is unchanged.

In [ ]:
def downsample_multinomial_orig(counts, target, rng):
    counts = np.asarray(counts, dtype=np.int64)
    total = int(counts.sum())
    if total <= target:
        return counts.copy()
    return rng.multinomial(target, counts / total)


def load_sample_umap_orig(df_s, vgene, model, emb_dict, target_depth, seed):
    df_s = df_s.copy()
    df_s["v_gene"] = df_s["v_gene"].astype(str)
    df_s["count"] = pd.to_numeric(df_s["count"], errors="coerce").fillna(0).astype(int)
    df_s = df_s[df_s["count"] > 0].reset_index(drop=True)

    df_s["clone_key"] = list(zip(df_s["cdr3aa"], df_s["v_gene"]))
    df_s["embedding"] = df_s["clone_key"].map(emb_dict)
    df_s = df_s.dropna(subset=["embedding"]).reset_index(drop=True)

    if len(df_s) < 2:
        return None, None

    rng = np.random.default_rng(seed)
    counts = df_s["count"].values.astype(np.int64)
    ds = downsample_multinomial_orig(counts, target_depth, rng)
    keep = ds > 0
    df_s = df_s.loc[keep].copy()
    df_s["ds"] = ds[keep]

    sub = df_s[df_s["v_gene"] == vgene]
    if len(sub) < 2:
        return None, None

    emb = np.vstack(sub["embedding"].values).astype(np.float32)
    w = sub["ds"].values.astype(np.float64)
    X_scaled = model["scaler"].transform(emb)
    X_umap = model["umap_reducer"].transform(X_scaled)
    return X_umap, w


xy_orig, w_orig = load_sample_umap_orig(df, V_GENE, reference[V_GENE], embeddings, TARGET_DEPTH, SEED)
n_orig = 0 if xy_orig is None else len(xy_orig)
print(f"[original-logic] {V_GENE}: {n_orig} clones")

## Step 5 — compare counts and coordinates numerically

In [ ]:
print(f"package n_clones      = {len(coords_pkg)}")
print(f"original-logic n_clones = {n_orig}")

if len(coords_pkg) == n_orig and n_orig > 0:
    xy_pkg = coords_pkg[["umap_1", "umap_2"]].values
    w_pkg = coords_pkg["weight"].values
    # Sort both by (x, y) so row order doesn't matter for the comparison
    order_pkg = np.lexsort((xy_pkg[:, 1], xy_pkg[:, 0]))
    order_orig = np.lexsort((xy_orig[:, 1], xy_orig[:, 0]))
    max_coord_diff = np.abs(xy_pkg[order_pkg] - xy_orig[order_orig]).max()
    max_weight_diff = np.abs(np.sort(w_pkg) - np.sort(w_orig)).max()
    print(f"max |coord diff| (sorted match) = {max_coord_diff:.6f}")
    print(f"max |weight diff| (sorted match) = {max_weight_diff:.6f}")
else:
    print("Clone counts differ -> the two paths are working from a different "
          "set of rows before we even get to plotting. Look at Step 1's raw "
          f"row count for {V_GENE} and re-check downsample/seed settings above.")

## Step 6 — render both side by side with identical plotting code

In [ ]:
def compute_kde_grid(xy, weights, xx, yy):
    if xy is None or len(xy) < 10:
        return None
    try:
        w = weights / weights.sum()
        kde = gaussian_kde(xy.T, bw_method="scott", weights=w)
        return kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    except Exception:
        return None


def render_panel(ax, xy, w, ref_umap, ref_w, color, title, grid_n=GRID_N):
    x0, x1 = np.percentile(ref_umap[:, 0], 1) - 1, np.percentile(ref_umap[:, 0], 99) + 1
    y0, y1 = np.percentile(ref_umap[:, 1], 1) - 1, np.percentile(ref_umap[:, 1], 99) + 1
    xx, yy = np.mgrid[x0:x1:complex(grid_n), y0:y1:complex(grid_n)]

    dens_ref = compute_kde_grid(ref_umap, ref_w, xx, yy)
    if dens_ref is not None:
        ax.contourf(xx, yy, dens_ref, levels=20, cmap="Greys", alpha=0.35, zorder=0)
        ax.contour(xx, yy, dens_ref, levels=[dens_ref.max() * t for t in [0.3, 0.6]],
                   colors="gray", linewidths=0.8, alpha=0.6, zorder=1)

    dens_q = compute_kde_grid(xy, w, xx, yy)
    if dens_q is not None:
        valid = dens_q[dens_q > 0]
        if len(valid) > 4:
            levels_q = np.percentile(valid, [50, 70, 85, 95])
            ax.contourf(xx, yy, dens_q, levels=[levels_q[0], dens_q.max()], colors=[color], alpha=0.15, zorder=2)
            ax.contour(xx, yy, dens_q, levels=levels_q, colors=[color], linewidths=1.6, alpha=0.9, zorder=3)

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_title(f"{title} (n={len(xy) if xy is not None else 0})", fontsize=11, color=color)


ref_umap = reference[V_GENE]["ref_umap2d"]
ref_w = reference[V_GENE].get("ref_ds_weights", np.ones(len(ref_umap)))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
render_panel(axes[0], coords_pkg[["umap_1", "umap_2"]].values, coords_pkg["weight"].values,
             ref_umap, ref_w, "#1565C0", f"package: {V_GENE}")
render_panel(axes[1], xy_orig, w_orig, ref_umap, ref_w, "#C62828", f"original-logic: {V_GENE}")
plt.tight_layout()
plt.show()

## Reading the result

- If Step 5 shows `n_clones` matching and near-zero max diffs, and Step 6's two panels look the same: the package logic is correct,
  and any earlier visual mismatch you saw was from comparing against a different sample/timepoint/reference than you thought.
- If `n_clones` differs here too: paste the printed counts and I'll dig into the downsampling call itself — since this notebook now
  eliminates every other variable (same df, same reference, same embeddings), the only remaining place a difference could come from
  is the exact downsample/seed handling inside `plot_umap` itself.